# Step 2: Scale with SageMaker AI jobs

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 4.6.0</strong> and the <strong>SageMaker Python SDK version 3.27.0</strong>.
</div>

In this step you move the data processing and model training from notebook 01 to [SageMaker AI jobs](https://docs.aws.amazon.com/sagemaker/latest/dg/train-model.html). A job runs your code in a container on managed instances, which SageMaker AI starts for the job and stops when the job ends. You use the [SageMaker Python SDK](https://sagemaker.readthedocs.io/en/stable/) to start the jobs and the container image that you built in notebook 00.

||||
|---|---|---|
|1. |Experiment in a notebook ||
|2. |Scale with SageMaker AI jobs and SageMaker SDK |**<<<< YOU ARE HERE**|
|3. |Operationalize with ML pipeline, model registry ||
|4. |Deploy the model and add data and model monitoring ||
|5. |Automate model deployment with a SageMaker project ||

The processing and training code is in reusable Python functions in the [`pipeline_steps`](pipeline_steps/) folder. You use the same functions in this notebook and in the ML pipeline in step 3. In this notebook you call them in the notebook first, and then run them on SageMaker AI in two ways:

| | [`@remote` decorator](https://docs.aws.amazon.com/sagemaker/latest/dg/train-remote-decorator.html) | [`ModelTrainer`](https://sagemaker.readthedocs.io/en/stable/training/index.html) in script mode |
|---|---|---|
| **What runs** | A Python function | A Python file, the entry script of the container |
| **Inputs and outputs** | Function arguments and return value | Input channels in `/opt/ml/input/data`, model artifacts in `/opt/ml/model` |
| **You configure** | Instance type, image, environment | Every setting of the training job: image, source code, channels, hyperparameters, output, compute |
| **Use it for** | Moving notebook code to SageMaker AI with few changes | Production training jobs and training at scale |
| **In a pipeline (step 3)** | `@step` decorator | Native `TrainingStep` |

![The train function runs in the notebook, with remote, with ModelTrainer, and with step](img/one-function-four-ways.svg)

At the end of the notebook you register the model that the `ModelTrainer` job trained, so that you can deploy it, and you explain it with a third function, `explain()`, which runs with `@remote`.

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), (
    f"This workshop needs Python 3.12, but the kernel runs Python {sys.version_info.major}.{sys.version_info.minor}. "
    "Run ./setup-py312.sh in a terminal, refresh the browser, and select the Python 3.12 kernel. "
    "See 00-start-here.ipynb for the steps."
)

Load the variables that the previous notebooks stored.

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

In [ ]:
import warnings
import os
import time
import inspect
from html import escape
from IPython.display import HTML
import tempfile
import shutil
import boto3
import mlflow
import sagemaker_mlflow
from mlflow import MlflowClient
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn import metrics
import matplotlib.pyplot as plt

from sagemaker.core.utils.utils import configure_logging
from sagemaker.core.common_utils import name_from_base
from sagemaker.core.s3.client import s3_path_join
from sagemaker.core.shapes import OutputDataConfig
from sagemaker.core.resources import ModelPackage
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.remote_function.client import remote
from sagemaker.core.remote_function.custom_file_filter import CustomFileFilter
from sagemaker.train.configs import InputData, SourceCode, Compute
from sagemaker.train import ModelTrainer
from sagemaker.train.model_trainer import Mode

# Reusable functions shared with the ML pipeline in notebook 03
from pipeline_steps.preprocess import preprocess
from pipeline_steps.train_fn import train

In [ ]:
warnings.filterwarnings('ignore')
configure_logging('INFO')

---

## Set the MLflow experiment
Reuse the experiment from notebook 01. This notebook tracks its runs in the same experiment.

In [ ]:
# The experiment name is restored from %store (created in notebook 01).
# Fall back to the same date-suffixed name if this notebook is run standalone.
if "experiment_name" not in dir():
    import time
    experiment_name = f"from-idea-to-prod-experiment-{time.strftime('%d%m%y', time.gmtime())}"
    %store experiment_name

print(f"Using MLflow experiment: {experiment_name}")

In [ ]:
# MLflow prints "View run ... at: https://mlflow.sagemaker.<region>.app.aws/..." when a run
# ends. That host is the SageMaker MLflow *tracking API* endpoint, not the UI, so the links
# do not open. Suppress them; the notebook shows working presigned UI links instead.
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "true"
mlflow.set_tracking_uri(mlflow_arn)
experiment = mlflow.set_experiment(experiment_name=experiment_name)
print(f'Using MLflow server: {mlflow_arn}')

---

## Reusable functions for processing and training
The folder [`pipeline_steps`](pipeline_steps/) contains the data processing and model training code from notebook 01 as plain Python functions:
- `preprocess()` in [`pipeline_steps/preprocess.py`](pipeline_steps/preprocess.py) runs the feature engineering, splits the data, and writes the train, validation, test, and baseline datasets to S3
- `train()` in [`pipeline_steps/train_fn.py`](pipeline_steps/train_fn.py) trains the XGBoost model and logs it to MLflow

Both functions track their parameters, metrics, and artifacts in MLflow. They read the MLflow app and the experiment from the environment variables `MLFLOW_TRACKING_URI` and `MLFLOW_EXPERIMENT_NAME`, so the same code works in the notebook and in a SageMaker AI job.

The next cell defines a helper that shows the source code of a function in the notebook.

In [ ]:
def source_details(function):
    """Return a collapsed, HTML-escaped view of a function's source code."""
    qualified_name = f"{function.__module__}.{function.__name__}"
    signature = escape(str(inspect.signature(function)))
    source = escape(inspect.getsource(function))
    return HTML(
        f"""
        <details style="margin: 0.5rem 0 1rem; border: 1px solid var(--jp-border-color2, #ddd); border-radius: 6px; padding: 0.6rem 0.8rem;">
          <summary style="cursor: pointer; font-weight: 600;">
            View source: <code>{escape(qualified_name)}{signature}</code>
          </summary>
          <pre style="margin-top: 0.8rem; max-height: 32rem; overflow: auto; white-space: pre;"><code>{source}</code></pre>
        </details>
        """
    )

### Settings for `@remote` jobs
The [`@remote` decorator](https://docs.aws.amazon.com/sagemaker/latest/dg/train-remote-decorator.html) serializes a Python function and its arguments, runs the function as a SageMaker training job, and returns the result to the notebook. The job runs on the container image that you built in notebook 00, which contains all dependencies.

`include_local_workdir=True` uploads the workshop folder to the job, so that the job can import the `pipeline_steps` module. The file filter excludes data, images, and notebooks from the upload.

All jobs in this notebook, processing and training, run on the same instance type `instance_type`.

In [ ]:
# One instance type and count for all processing and training jobs in this notebook
instance_type = "ml.r5.xlarge"
instance_count = 1

region = Session().boto_region_name
output_s3_prefix = s3_path_join("s3://", bucket_name, bucket_prefix)

# MLflow reads the first two variables natively. pipeline_steps/runtime_utils.py logs CONTAINER_IMAGE as a run tag.
job_environment = {
    "MLFLOW_TRACKING_URI": mlflow_arn,
    "MLFLOW_EXPERIMENT_NAME": experiment_name,
    "CONTAINER_IMAGE": xgboost_image,
    "USER": user_profile_name,
    "REGION": region,
}

remote_settings = {
    "instance_type": instance_type,
    "role": sm_role,
    "image_uri": xgboost_image,
    "include_local_workdir": True,
    "custom_file_filter": CustomFileFilter(
        ignore_name_patterns=[
            "data", "tmp", "img", "design", "additional-topics",
            "deployment-seed-code", "processing", "training", "*.ipynb", "__pycache__", ".git",
        ]
    ),
    "environment_variables": job_environment,
    "keep_alive_period_in_seconds": 3600,
}

# The functions read the same variables when you call them in the notebook.
# CONTAINER_IMAGE=local marks these runs as local in MLflow.
os.environ.update({**job_environment, "CONTAINER_IMAGE": "local"})

print(f"Instance: {instance_type}")
print(f"Image:    {xgboost_image}")
print(f"Output:   {output_s3_prefix}")

---

## Process the data

### Inspect `preprocess()`
Expand the panel to see the function. It reuses the feature engineering from notebook 01 in `engineer_features()`, splits the data, and writes the datasets as header-less CSV files with the label in the first column. It also writes the whole dataset without the label column as a baseline dataset, which you need for model monitoring in step 4.

In [ ]:
display(source_details(preprocess))

### Validate `preprocess()` locally
Call the function in the notebook with the local dataset and a local output folder. This is a quick check before you start a SageMaker job.

In [ ]:
local_output_prefix = os.path.abspath("tmp/local-validation")
for split in ["train", "validation", "test", "baseline"]:
    os.makedirs(os.path.join(local_output_prefix, split), exist_ok=True)

local_preprocess_result = preprocess(
    input_data_s3_path=dataset_file_local_path,
    output_s3_prefix=local_output_prefix,
    pipeline_run_name=name_from_base("notebook-02-local", short=True),
)
local_preprocess_result

### Run `preprocess()` on SageMaker AI
Wrap the same function with `remote()` and call it with the dataset in S3. The function runs as a SageMaker training job and returns the same dictionary as the call in the notebook.

The function also creates a parent MLflow run. You pass its ID to the training functions in the next sections, so that MLflow groups all runs of this notebook under one parent run.

In [ ]:
remote_preprocess = remote(**remote_settings, job_name_prefix="from-idea-to-prod-preprocess")(preprocess)

preprocess_result = remote_preprocess(
    input_data_s3_path=s3_data_uri,
    output_s3_prefix=output_s3_prefix,
    pipeline_run_name=name_from_base("notebook-02", short=True),
)
preprocess_result

Store the S3 locations of the datasets for the next notebooks.

In [ ]:
train_s3_url = preprocess_result["train_data"].rsplit("/", 1)[0]
validation_s3_url = preprocess_result["validation_data"].rsplit("/", 1)[0]
test_s3_url = preprocess_result["test_x_data"].rsplit("/", 1)[0]
baseline_s3_url = preprocess_result["baseline_data"].rsplit("/", 1)[0]

%store train_s3_url
%store validation_s3_url
%store test_s3_url
%store baseline_s3_url

Show the first rows of the training data. The first column is the label.

In [ ]:
pd.read_csv(preprocess_result["train_data"], header=None).head()

---

## Train the model
You train the model three times: in the notebook to check the code, with `@remote`, and with `ModelTrainer`. All three runs call the same `train()` function, so they produce comparable models and appear side by side in MLflow.

SageMaker AI offers four ways to train a model. This workshop uses its own container, which you built in notebook 00.

![SageMaker AI training options: built-in algorithms, script mode, your own container, and AWS Marketplace](img/sagemaker-training-options.svg)

Set the hyperparameters. `train()` accepts them as keyword arguments.

In [ ]:
hyperparams = {
    'num_round': 50,
    'max_depth': 3,
    'eta': 0.5,
    'alpha': 2.5,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'early_stopping_rounds': 10,
    'verbosity': 1
}

### Inspect `train()`

In [ ]:
display(source_details(train))

### Validate `train()` locally
Train in the notebook on the datasets that the processing job wrote to S3.

In [ ]:
local_training_result = train(
    train_data_s3_path=preprocess_result["train_data"],
    validation_data_s3_path=preprocess_result["validation_data"],
    output_s3_prefix=local_output_prefix,
    pipeline_run_id=preprocess_result["pipeline_run_id"],
    **hyperparams,
)
print(f"Local validation AUC: {local_training_result['validation_auc']:.4f}")

### Train with the `@remote` decorator
Wrap `train()` with `remote()`, the same way you did for `preprocess()`. This is the quickest way to run a Python function on SageMaker AI managed infrastructure.

In [ ]:
remote_train = remote(**remote_settings, job_name_prefix="from-idea-to-prod-remote-train")(train)

remote_training_result = remote_train(
    train_data_s3_path=preprocess_result["train_data"],
    validation_data_s3_path=preprocess_result["validation_data"],
    output_s3_prefix=output_s3_prefix,
    pipeline_run_id=preprocess_result["pipeline_run_id"],
    **hyperparams,
)
print(f"Remote validation AUC: {remote_training_result['validation_auc']:.4f}")

---

### Train with `ModelTrainer` in script mode
The [`ModelTrainer`](https://sagemaker.readthedocs.io/en/stable/training/index.html) class gives you control over every setting of the training job: the training image, the entry script, the input channels, the hyperparameters, the output location, and the compute. Use it for production training jobs, for SageMaker AI features such as Spot instances, checkpoints, and distributed training, and to run a SageMaker AI built-in or framework image.

`ModelTrainer` runs a Python file as the entry script in the container, instead of a function. You use the same file [`pipeline_steps/train_fn.py`](pipeline_steps/train_fn.py). Its `main()` function at the end of the file reads the hyperparameters and the input channels that SageMaker AI passes to the script, and calls the same `train()` function. In step 3 a native `TrainingStep` runs this `ModelTrainer` in a pipeline.

Define the input channels. SageMaker downloads the content of each S3 location to `/opt/ml/input/data/<channel_name>` in the container.

In [ ]:
training_input_data = [
    InputData(channel_name="train", data_source=train_s3_url),
    InputData(channel_name="validation", data_source=validation_s3_url),
]

output_s3_url = f"s3://{bucket_name}/{bucket_prefix}/output"
%store output_s3_url

Define the `ModelTrainer` with the container image, the source code, the compute, the hyperparameters, and the environment variables. `SourceCode` uploads only the `pipeline_steps` folder and runs `train_fn.py`. `Mode.SAGEMAKER_TRAINING_JOB` runs the job on SageMaker AI managed infrastructure.

In [ ]:
trainer_managed = ModelTrainer(
    training_image=xgboost_image,
    source_code=SourceCode(
        source_dir="pipeline_steps",
        entry_script="train_fn.py",
        ignore_patterns=["__pycache__", ".ipynb_checkpoints"],
    ),
    compute=Compute(
        instance_type=instance_type,
        instance_count=instance_count
    ),
    hyperparameters={
        **hyperparams,
        "output_s3_prefix": output_s3_prefix,
        "pipeline_run_id": preprocess_result["pipeline_run_id"],
    },
    base_job_name="from-idea-to-prod-training",
    training_mode=Mode.SAGEMAKER_TRAINING_JOB,
    output_data_config=OutputDataConfig(s3_output_path=output_s3_url),
    environment=job_environment,
)

In [ ]:
# train() waits until the job completes and raises an error if the job fails
trainer_managed.train(
    input_data_config=training_input_data,
    wait=True,
    logs=False,
)

The training job saved the model artifacts to S3.

In [ ]:
# ModelTrainer keeps the last training job it started. The SDK has no public accessor for it yet.
training_job = trainer_managed._latest_training_job
training_job_name = training_job.training_job_name

print(f"Training job:    {training_job_name}")
print(f"Model artifacts: {training_job.model_artifacts.s3_model_artifacts}")

In [ ]:
display(
    HTML('<b>See the SageMaker <a target="_blank" rel="noopener noreferrer" href="https://{}.console.aws.amazon.com/sagemaker/home?region={}#/jobs/{}">training job</a></b>'.format(
            region, region, training_job_name))
)

---

## Register the model from the `ModelTrainer` job
Register the model that the `ModelTrainer` job trained. Because notebook 00 created the MLflow app with `AutoModelRegistrationEnabled`, each `mlflow.register_model()` call also creates a model package version in the [SageMaker Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html). You register in one place only.

A model package is *deployable* only if it has an [inference specification](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-track-experiments-model-registration.html): the serving container image, the location of the model data, and the environment variables of the serving code. Before you register the model, you:

1. Log the `inference.py` serving code into the artifact store of the logged model, under `code/`, with `MlflowClient.log_model_artifacts()`
2. Log the inference specification on the logged model with the `sagemaker-mlflow` plugin
3. Call `mlflow.register_model()`. The automatic registration copies the specification to the model package.

![Log inference.py and the inference specification, register the model in MLflow, sync it to the SageMaker Model Registry, deploy it to an endpoint that reads the model files from the MLflow artifact store](img/register-deployable-model.svg)

The MLflow artifact store stays the single source of truth for the model files: no repacking and no copies.

<div class="alert alert-warning">
<b>Order matters:</b> log the specification <b>before</b> you call <code>mlflow.register_model()</code>. The automatic registration copies the specification when it creates the model package. If you log it afterwards, the model package exists already, without the specification.
</div>

In [ ]:
# Reuse the single registered model name shared across all notebooks
# (created in notebook 01, restored via %store). New registrations become new
# VERSIONS of the same registered model instead of a new model per notebook.
if "registered_model_name" not in dir():
    import time
    registered_model_name = f"from-idea-to-prod-model-{time.strftime('%d%m%y', time.gmtime())}"
    %store registered_model_name

print(f"Registering to model: {registered_model_name}")

Find the MLflow run of the `ModelTrainer` job and its **logged model**: the MLflow 3 entity that carries the `model_id` and `artifact_location` the inference specification points at. `train()` tags each run with the SageMaker job name, and it runs `mlflow.xgboost.autolog(..., log_models=False)` and logs the model explicitly with `mlflow.xgboost.log_model()`, which is what makes it a searchable logged model on that run.

In [ ]:
mlflow_client = MlflowClient()
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# pipeline_steps/runtime_utils.py tags every run with the SageMaker job name
training_runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string=f"tags.`sagemaker.job_name` = '{training_job_name}'",
)
if training_runs.empty:
    raise RuntimeError(f"No MLflow run found for the training job {training_job_name}")
last_run_id = training_runs["run_id"][0]

# Resolve the MLflow 3 logged model produced by that run
logged_model = next(
    m for m in mlflow.search_logged_models(
        experiment_ids=[experiment_id], output_format="list"
    )
    if m.source_run_id == last_run_id
)

print(f"Source run:        {last_run_id}")
print(f"Logged model id:   {logged_model.model_id}")
print(f"Artifact location: {logged_model.artifact_location}")

### Understand `inference.py`
A SageMaker AI endpoint runs your container with `docker run <image> serve`. The container must run a web server on port 8080 that answers two requests ([container contract](https://docs.aws.amazon.com/sagemaker/latest/dg/your-algorithms-inference-code.html)):
- `GET /ping`: the health check. SageMaker AI sends traffic to the instance only after `/ping` returns `200`.
- `POST /invocations`: the inference request, with the request body and the `Content-Type` and `Accept` headers that the client sends

You don't write this web server. The SageMaker AI framework containers include one, and the image of this workshop includes [`byoc/serving_app.py`](byoc/serving_app.py). You write only `inference.py`, a Python file with up to four handler functions. The server imports the file and calls the functions:

![The serving app calls model_fn once at startup, then input_fn, predict_fn, and output_fn for each request](img/inference-handlers.svg)

|Function|When the server calls it|Input|Returns|In this workshop|
|---|---|---|---|---|
|`model_fn(model_dir)`|Once, when the container starts|The model folder `/opt/ml/model`|The model object|Loads the booster with `mlflow.xgboost.load_model()`|
|`input_fn(request_body, request_content_type)`|For each request|The request body and its `Content-Type`|The model input|Parses CSV or JSON into a 2D numpy array of floats|
|`predict_fn(input_data, model)`|For each request|The output of `input_fn` and the model|The prediction|Calls `model.predict()`, which returns the probability of `yes`|
|`output_fn(prediction, accept)`|For each request|The output of `predict_fn` and the `Accept` header|The response body and its content type|One probability per line for CSV, or `{"predictions": [...]}` for JSON|

The SageMaker AI framework containers for PyTorch, scikit-learn, and XGBoost use the same four function names, so what you learn here applies to them too. The serving app of this workshop has defaults for `input_fn`, `predict_fn`, and `output_fn`, but no default for `model_fn`: without `model_fn` the container fails the `/ping` health check and the endpoint never reaches `InService`.

**How the container finds the model and the script.** The inference specification that you log in the next sections sets three things:
- the model data source: SageMaker AI downloads everything under the artifact location of the logged model to `/opt/ml/model`
- `SAGEMAKER_SUBMIT_DIRECTORY=/opt/ml/model/code`: the folder of the script
- `SAGEMAKER_PROGRAM=inference.py`: the name of the script

So the endpoint sees this layout:

```
/opt/ml/model/                  <- model_dir in model_fn
├── MLmodel                     <- MLflow model files, read by mlflow.xgboost.load_model()
├── model.ubj
├── conda.yaml
├── python_env.yaml
├── requirements.txt
├── ...                         <- more MLflow metadata files
└── code/                       <- SAGEMAKER_SUBMIT_DIRECTORY
    └── inference.py            <- SAGEMAKER_PROGRAM
```

Expand the panels to see the four functions in [`pipeline_steps/inference.py`](pipeline_steps/inference.py).

In [ ]:
import pipeline_steps.inference as inference

for handler in [inference.model_fn, inference.input_fn, inference.predict_fn, inference.output_fn]:
    display(source_details(handler))

### Log `inference.py` into the artifact store of the model
The register step of the ML pipeline in step 3 logs the same file. The next cell logs it with MLflow instead of a direct S3 upload, so the artifact store stays the single source of truth and the script shows up in the MLflow UI next to the model.

In [ ]:
# log_model_artifacts keeps the directory layout, so code/inference.py
# lands at <artifact_location>/code/inference.py
with tempfile.TemporaryDirectory() as tmp:
    os.makedirs(os.path.join(tmp, "code"))
    shutil.copy("pipeline_steps/inference.py", os.path.join(tmp, "code", "inference.py"))
    mlflow_client.log_model_artifacts(logged_model.model_id, tmp)

print(f"Logged code/inference.py to {logged_model.artifact_location}/code/")

### Call the handlers in the notebook
Before you deploy anything, call the handlers in the same order as the serving app. The next cell downloads the logged model to a local folder, which has the same layout as `/opt/ml/model` on the endpoint, and sends three test records through `input_fn`, `predict_fn`, and `output_fn`.

In [ ]:
# Same files that the endpoint downloads to /opt/ml/model
local_model_dir = mlflow.artifacts.download_artifacts(artifact_uri=f"models:/{logged_model.model_id}")
for root, _, files in sorted(os.walk(local_model_dir)):
    folder = os.path.relpath(root, local_model_dir)
    print(f"{'model_dir' if folder == '.' else 'model_dir/' + folder}/: {', '.join(sorted(files))}")

# 1. At startup: load the model
served_model = inference.model_fn(local_model_dir)

# 2. For each request: parse, predict, serialize
request_body = pd.read_csv(preprocess_result["test_x_data"], header=None, nrows=3).to_csv(header=False, index=False)
input_data = inference.input_fn(request_body, "text/csv")
prediction = inference.predict_fn(input_data, served_model)
response_body, content_type = inference.output_fn(prediction, "text/csv")

print(f"\ninput_fn   -> {type(input_data).__name__} with shape {input_data.shape}")
print(f"predict_fn -> {prediction}")
print(f"output_fn  -> {content_type}:\n{response_body}")

### Log the inference specification

`sagemaker_mlflow.log_inference_specification()` attaches the container image, the model data source, and the environment variables to the logged model. The model package can only be deployed on the instance types that you list here, so include the type that you plan to use.

In [ ]:
inference_spec = {
    "Containers": [{
        "Image": xgboost_image,
        # Serve straight from the MLflow artifact store - no repacking, no copies.
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": logged_model.artifact_location.rstrip("/") + "/",
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
        # Point the BYOC serving stack at the script logged above.
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        },
    }],
    "SupportedContentTypes": ["text/csv", "application/json"],
    "SupportedResponseMIMETypes": ["text/csv", "application/json"],
    "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large", "ml.m5.xlarge", "ml.r5.xlarge"],
    "SupportedTransformInstanceTypes": ["ml.m5.large", "ml.m5.xlarge"],
}

sagemaker_mlflow.log_inference_specification(
    logged_model.model_id, inference_specification=inference_spec
)
print("Inference specification logged on the MLflow model.")

### Register the model
Register the logged model. The sync to the SageMaker Model Registry runs asynchronously. It adds the ARN of the model package as the tag `sagemaker.model_package_arn` to the MLflow model version, so you don't need to search the SageMaker Model Registry for the package.

In [ ]:
registered_model_version = mlflow.register_model(
    f"models:/{logged_model.model_id}", registered_model_name
)
print(f"Registered in MLflow: {registered_model_version.name} v{registered_model_version.version}")

model_package_arn = None
for _ in range(20):
    mv = mlflow_client.get_model_version(
        registered_model_version.name, registered_model_version.version
    )
    model_package_arn = mv.tags.get("sagemaker.model_package_arn")
    if model_package_arn:
        break
    print(".", end="", flush=True)
    time.sleep(3)

if not model_package_arn:
    raise TimeoutError("SageMaker Model Package was not auto-created within the timeout")

print(f"\nSynced SageMaker Model Package: {model_package_arn}")

### Inspect the model package
The model package has an `InferenceSpecification`, because you logged it before you registered the model. The next cell reads the specification from the SageMaker Model Registry and fails if the environment variables are missing. Without `SAGEMAKER_PROGRAM` the container has no inference code to run.

In [ ]:
model_package = ModelPackage.get(model_package_name=model_package_arn)

# DescribeModelPackage does not return ModelPackageName for versioned packages,
# which breaks refresh()/update(). The API accepts an ARN in that field, so
# backfill it here.
model_package.model_package_name = model_package.model_package_arn

# The typed ContainersItem shape omits Environment and friends, so read the raw
# DescribeModelPackage response for those.
described = boto3.client("sagemaker").describe_model_package(
    ModelPackageName=model_package_arn
)
raw_container = described["InferenceSpecification"]["Containers"][0]

print(f"Model package group:  {model_package.model_package_group_name}")
print(f"Version:              {model_package.model_package_version}")
print(f"Status:               {model_package.model_package_status}")
print(f"Approval status:      {described.get('ModelApprovalStatus', '(not set)')}")

print("\nInference specification, as stored in the registry")
print(f"  image:              {raw_container['Image']}")
print(f"  model data source:  {raw_container['ModelDataSource']['S3DataSource']['S3Uri']}")
print(f"  environment:        {raw_container.get('Environment', {})}")

expected_env = inference_spec["Containers"][0]["Environment"]
synced_env = raw_container.get("Environment", {})
mismatched = {k: (v, synced_env.get(k)) for k, v in expected_env.items() if synced_env.get(k) != v}
if mismatched:
    raise RuntimeError(
        f"Environment did not round-trip through the MLflow sync: {mismatched}. "
        "The endpoint would fall back to the container's defaults."
    )
print("\nEnvironment round-tripped through the sync intact.")

### View the logged model in the MLflow UI
Open the MLflow UI with a presigned URL, then open the artifacts of the logged model. `code/inference.py` is next to the model files. The endpoint downloads exactly these files to `/opt/ml/model`.

In [ ]:
sm_client = boto3.Session().client("sagemaker")

experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# get the presigned url to open the MLflow UI
presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn,
    ExpiresInSeconds=300,
    SessionExpirationDurationInSeconds=1800
)['AuthorizedUrl']

logged_model_artifacts = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment_id}/models/{logged_model.model_id}/artifacts?workspace=default"

In [ ]:
# Open the MLflow UI first: the presigned URL authenticates your browser session
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(presigned_url, quote=True)}">'
    '1. Open the MLflow UI and authenticate</a></b>'
))

In [ ]:
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(logged_model_artifacts, quote=True)}">'
    '2. Open the logged model artifacts in MLflow</a></b>'
))

Print the metrics and metadata of the registered model.

In [ ]:
print(f"Model: {registered_model_version.name}, version: {registered_model_version.version}")
print(f"Tags: {registered_model_version.tags}")
print(f"Aliases: {registered_model_version.aliases}")
print(f"\nMetrics:")
mlflow.get_run(logged_model.source_run_id).data.metrics

---

## Explain the model
Before you deploy a model, check *why* it predicts what it predicts, and whether it treats groups of clients differently. The function `explain()` in [`pipeline_steps/explain.py`](pipeline_steps/explain.py) does both, with open-source code only:
- **Feature attribution** with [SHAP](https://shap.readthedocs.io/). For a tree model like XGBoost, `shap.TreeExplainer` computes the exact contribution of each feature to each prediction, for the whole test dataset in about a second.
- **Fairness metrics** for a group of clients, the *facet*. Here the facet is the clients aged 60 and older, the one-hot columns `age_60-69` and `age_70-plus`. The function compares the facet with all other clients with standard bias metrics: class imbalance (CI), the difference in positive proportions in the labels (DPL) and in the predictions (DPPL), and the disparate impact (DI).

`explain()` logs the plots, a table with the SHAP values of each prediction, and the metrics in a new MLflow run, and copies the report to S3. In step 3 the same function is a pipeline step.

### Inspect `explain()`

In [ ]:
from pipeline_steps.explain import explain

display(source_details(explain))

### Run `explain()` on SageMaker AI
The container image doesn't include `shap` and `matplotlib`. The `dependencies` setting of `@remote` installs the packages in [`explainability/requirements.txt`](explainability/requirements.txt) when the job starts, only for this job. You don't need to rebuild the image.

The job explains the model of the `ModelTrainer` run that you registered.

In [ ]:
explain_settings = {
    **remote_settings,
    # shap and matplotlib aren't in the container image: install them for this job only
    "dependencies": "./explainability/requirements.txt",
    "job_name_prefix": "from-idea-to-prod-explain",
}
remote_explain = remote(**explain_settings)(explain)

explain_result = remote_explain(
    mlflow_run_id=last_run_id,
    test_x_data_s3_path=preprocess_result["test_x_data"],
    test_y_data_s3_path=preprocess_result["test_y_data"],
    feature_names=preprocess_result["feature_names"],
    output_s3_prefix=output_s3_prefix,
    facet_columns=["age_60-69", "age_70-plus"],
    pipeline_run_id=preprocess_result["pipeline_run_id"],
)

In [ ]:
print(f"Explained model: {explain_result['explained_model_id']}")
print(f"Report in S3:    {explain_result['report_s3_prefix']}")

print("\nTop features by mean |SHAP|:")
for rank, feature in enumerate(explain_result["top_features"][:10], 1):
    print(f"  {rank:<3}{feature}")

print("\nFairness metrics for the clients aged 60+:")
for name, value in explain_result["bias_metrics"].items():
    print(f"  {name:<40}{value:.4f}" if isinstance(value, float) else f"  {name:<40}{value}")

### Interpret the report
- **Top features**: the features with the largest mean absolute SHAP value move the predictions most. Check that they make business sense. A feature that leaks the label, like the call duration that notebook 01 removed, would show up at the top.
- **Size of the facet**: check `n_facet_group` before you read the ratios. The facet has fewer than 100 clients in the test dataset, so the metrics are noisy.
- **DPL and DPPL**: negative values mean that the facet has a *higher* positive rate than the other clients, in the labels (DPL) and in the predictions (DPPL). If DPPL is close to DPL, the model reproduces the difference that is already in the data and doesn't add to it.
- **DI**: the ratio of the positive prediction rates, facet divided by the other clients. A value of 1 means parity. A common rule of thumb flags values below 0.8. Here DI is far above 1, because older clients subscribe much more often in this dataset.

Whether a difference is acceptable is a business and legal decision, not a technical one. The report gives you the numbers for this decision.

Run the next cell and choose the link to open the explainability run with the SHAP plots in MLflow. Open the MLflow UI first with the link in the previous section, if the presigned URL expired.

In [ ]:
explain_artifacts_link = (
    f"{presigned_url.split('/auth')[0]}"
    f"/#/experiments/{experiment_id}/runs/{explain_result['explainability_run_id']}/artifacts?workspace=default"
)
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(explain_artifacts_link, quote=True)}">'
    'Open the explainability report in MLflow</a></b>'
))

---

## Test the registered model in the notebook
Check the registered model on the test dataset in the notebook. In [step 4](04-deployment-and-monitoring.ipynb) you deploy an approved model to a real-time endpoint and monitor it.

### Prepare the test data
Download the prepared test dataset from S3 to the notebook.

In [ ]:
!aws s3 cp $test_s3_url/test_x.csv tmp/test_x.csv
!aws s3 cp $test_s3_url/test_y.csv tmp/test_y.csv

### Run inference in the notebook
Load the registered model from the MLflow Model Registry and predict the test dataset.

In [ ]:
model_uri = registered_model_version.source
model = mlflow.xgboost.load_model(model_uri)

In [ ]:
test_x = np.loadtxt("tmp/test_x.csv", delimiter=",")
test_y = np.loadtxt("tmp/test_y.csv", delimiter=",")

dtest = xgb.DMatrix(test_x)
predictions = np.array(model.predict(dtest), dtype=float).squeeze()
predictions

In [ ]:
pd.crosstab(
    index=test_y,
    columns=np.round(predictions), 
    rownames=['actuals'], 
    colnames=['predictions']
)

### Log the confusion matrix in MLflow
Log the confusion matrix of the test dataset in the MLflow run of the model.

In [ ]:
def plot_confusion_matrix(
    cm, class_names, title="Confusion matrix", cmap=plt.cm.Blues, normalize=False
):
    if normalize:
        cm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        ylim=(cm.shape[0] - 0.5, -0.5),
        xticklabels=class_names,
        yticklabels=class_names,
        title=title,
        ylabel="Ground truth label",
        xlabel="Predicted label",
    )
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")
    fmt = ".2f"
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax, fig

In [ ]:
class_names = ["no", "yes"]
confusion_matrix = metrics.confusion_matrix(test_y, np.round(predictions))
ax, fig = plot_confusion_matrix(confusion_matrix, class_names)

In [ ]:
print(f"Log confusion matrix to the model {registered_model_version.name} version {registered_model_version.version}")
mlflow.set_experiment(experiment_name)
with mlflow.start_run(run_id=logged_model.source_run_id):
    mlflow.log_figure(fig, "confusion_matrix.png")

---

## Explore the experiments and the model registry in the MLflow UI
The MLflow UI shows all logged metrics, parameters, and artifacts. Run the next cell and choose the link, or open the MLflow UI in Studio: choose **MLflow** in the **Applications** pane, select the MLflow app, and choose **Open MLflow**.

![](img/mlflow-open.png)

In the MLflow UI you can browse the experiments, the runs, and the registered models:

![](img/experiment-mlflow-2.png)

![](img/models-mlflow-2.png)

![](img/model-fig-mlflow.png)

In [ ]:
# The presigned URL created earlier has been redeemed, so create a fresh one
presigned_url = boto3.Session().client("sagemaker").create_presigned_mlflow_app_url(
    Arn=mlflow_arn,
    ExpiresInSeconds=300,
    SessionExpirationDurationInSeconds=1800
)['AuthorizedUrl']

display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(presigned_url, quote=True)}">'
    'Open the MLflow UI</a></b>'
))

---

## Summary
In this notebook you moved the notebook code to SageMaker AI managed infrastructure:
- **Reusable functions**: the processing and training code is in the `preprocess()` and `train()` functions in `pipeline_steps`, which the ML pipeline in step 3 reuses
- **Processing with `@remote`**: you checked `preprocess()` in the notebook, then ran it as a SageMaker AI job
- **Training with `@remote`**: you ran `train()` as a SageMaker AI training job with one wrapper call
- **Training with `ModelTrainer`**: you ran the same `train()` code in script mode, with control over the container, the input channels, and the model artifacts
- **Registration**: you logged an inference specification on the MLflow model before you registered it, so the model package in the SageMaker Model Registry is deployable
- **Explainability**: you ran `explain()` with `@remote` and extra dependencies, to get the SHAP feature attributions and the fairness metrics of the model
- **Test**: you scored the registered model on the test dataset and logged the confusion matrix in MLflow

In step 3 you combine the same functions, including `explain()`, in an ML pipeline with SageMaker Pipelines.

## Continue with step 3
Open the step 3 notebook [03-sagemaker-pipeline](03-sagemaker-pipeline.ipynb).

## Further development ideas for your real-world projects
- Learn more about [managing experiments with MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- Try [SageMaker Canvas](https://docs.aws.amazon.com/sagemaker/latest/dg/canvas.html) to analyze your data and to build models with AutoML, without code
- Tune the hyperparameters with [automatic model tuning](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html). It runs many `ModelTrainer` training jobs for you
- Reduce the training cost with [managed Spot training](https://docs.aws.amazon.com/sagemaker/latest/dg/model-managed-spot-training.html) in `ModelTrainer`

## Additional resources
- [Using Docker containers with SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers.html)
- [Amazon SageMaker Immersion Day](https://catalog.us-east-1.prod.workshops.aws/workshops/63069e26-921c-4ce1-9cc7-dd882ff62575/en-US)
- [Amazon SageMaker Examples GitHub repository](https://github.com/aws/amazon-sagemaker-examples)
- [Amazon SageMaker Community Examples GitHub repository](https://github.com/aws/amazon-sagemaker-examples-community)

## Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>